In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

# This Notebook is deliberately safe by default. It runs the CPU Gate only.
# After independent review and separate explicit authorization, set RUN_BCD=True,
# enable Kaggle Internet, HF_TOKEN, and two visible T4 GPUs, then Run All once.
RUN_BCD = False
CODE_SOURCE_SHA = "7b3643461cead09285d2bbd1fdb417eb36597191"
REPOSITORY = "https://github.com/ArcheLabs/mini-cells.git"
CODE_ROOT = Path("/kaggle/working/r3mf-d1-code")
OUTPUT_ROOT = Path("/kaggle/working/r3mf-d1-output")


def run(command, cwd=None):
    print("[R3m-F-D1]", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), cwd=cwd, check=True)


if not importlib.util.find_spec("tokenizers"):
    run([sys.executable, "-m", "pip", "install", "--quiet", "tokenizers"])
if RUN_BCD and not importlib.util.find_spec("huggingface_hub"):
    run([sys.executable, "-m", "pip", "install", "--quiet", "huggingface_hub"])

if not (CODE_ROOT / ".git").exists():
    CODE_ROOT.mkdir(parents=True, exist_ok=True)
    run(["git", "init", "--quiet", str(CODE_ROOT)])
    run(["git", "-C", CODE_ROOT, "remote", "add", "origin", REPOSITORY])
    run(["git", "-C", CODE_ROOT, "config", "remote.origin.promisor", "true"])
    run(["git", "-C", CODE_ROOT, "config", "remote.origin.partialclonefilter", "blob:none"])
    run(["git", "-C", CODE_ROOT, "sparse-checkout", "init", "--no-cone"])
    run(["git", "-C", CODE_ROOT, "sparse-checkout", "set", "--no-cone", *["/pyproject.toml", "/src/", "/shared/python/", "/research/projects/cellular-language-models/engineering/kaggle-continuation/", "/research/projects/cellular-language-models/experiments/006-path-first-clm/", "/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/implementation/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r2/implementation/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3/implementation/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/run.py", "/research/projects/continual-learning/context/04-continual-learning/scripts/research/", "/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3d/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-v2/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3g/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3i/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3j/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3k/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/implementation/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/artifact-recovery-001/independent-review-002/tokenizer.json", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-e/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f/audits/tiny-failure-001/", "/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f-d1/"]])
    run(["git", "-C", CODE_ROOT, "fetch", "--depth=1", "--filter=blob:none", "origin", CODE_SOURCE_SHA])
    run(["git", "-C", CODE_ROOT, "checkout", "--detach", "FETCH_HEAD"])
else:
    current = subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True).strip()
    if current != CODE_SOURCE_SHA:
        raise RuntimeError("R3m-F-D1 code folder belongs to a different pinned source; start a fresh Kaggle session.")

actual = subprocess.check_output(["git", "-C", str(CODE_ROOT), "rev-parse", "HEAD"], text=True).strip()
if actual != CODE_SOURCE_SHA:
    raise RuntimeError("R3m-F-D1 source commit verification failed")
if subprocess.run(["git", "-C", str(CODE_ROOT), "diff", "--quiet", "HEAD", "--"]).returncode:
    raise RuntimeError("R3m-F-D1 source checkout contains uncommitted changes; use a clean checkout.")
print({"experiment": "CLM-001-R3m-F-D1-Supervision-Context-001", "source_sha": actual, "training_requested": RUN_BCD}, flush=True)

entry = CODE_ROOT / "research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f-d1/run.py"
command = [sys.executable, str(entry), "--mode", "auto", "--output", str(OUTPUT_ROOT)]
if RUN_BCD:
    command.append("--authorize-d1-run")
run(command, cwd=CODE_ROOT)
print("Save Version Output:", OUTPUT_ROOT, flush=True)
